# Demos of cosmological calculations

In [ ]:
import sys
from pathlib import Path

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import camb
from colossus.cosmology import cosmology

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from stepsic.parameters import CosmoParameters

In [ ]:
import logging
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
# Facecolor values from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#f4f0e8',
    'axes.facecolor': '#f4f0e8',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)
plt.rcParams['text.usetex'] = False

In [ ]:
params = CosmoParameters(path=Path.cwd() / 'config.toml').get_parameters()

## Cosmology

In [ ]:
def hubble_a(a, H0, omega_m, omega_l):
    r'''
    Computes the Hubble parameter :math:`H(a)` at scale factor :math:`a`.

    The Hubble parameter is given by

    .. math::
        H(a) = H_0\,\sqrt{\omega_m\,a^3 + (1 - \omega_m - \omega_\Lambda)\,a^2 + \omega_\Lambda},

    where :math:`a` is the scale factor normalized to 1 at present.
    :math:`H_0` is the Hubble constant, :math:`\omega_m` is the present-day
    matter density parameter and :math:`\omega_\Lambda` is the present-day
    dark energy density parameter.

    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    H0 : float
        Hubble constant in km/s/Mpc.
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The Hubble parameter :math:`H(a)`.
    '''
    return H0 * np.sqrt(omega_m / a**3 + (1 - omega_m - omega_l) / a**2 + omega_l)

In [ ]:
z = np.linspace(63, 0, 1000)
a = 1 / (1 + z)
H = hubble_a(a, params['H0'], params['OMEGA_M'], params['OMEGA_L'])

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

ax = axes[0]
ax.plot(z, H, color='tab:red', lw=3)

ax = axes[1]
ax.plot(z, H, color='tab:red', lw=3)
ax.scatter(z[-1], H[-1], color='black', marker='x', s=12**2, lw=2, zorder=2)
ax.axvline(0, color='0.7', lw=1, ls='--')
ax.axhline(params['H0'], color='green', lw=1, ls='--')
ax.text(0.05, 0.32, rf'$H_0 = {params["H0"]}\,\mathrm{{km/s/Mpc}}$', color='green',
        fontsize=10, ha='left', va='center', transform=ax.transAxes)
ax.set_ylim(60, 80)
ax.set_xlim(-0.2, 0.4)

for ax in axes:
    # ax.set_yscale('log')
    ax.invert_xaxis()
    ax.set_xlabel(r'Redshift $z$', fontsize=12)
    ax.set_title(r'Hubble parameter $H(z)$', loc='left', fontsize=10)

plt.show()

In [ ]:
def F_omega(a, omega_m, omega_l):
    r'''
    Computes the linear growth rate factor for first-order Lagrangian
    perturbation.

    This function returns the factor :math:`F_\omega(a)`, defined by

    .. math::
        F_\omega(a) = \left[\Omega(a)\right]^{0.6},

    where the effective matter density parameter :math:`\Omega(a)` is
    computed as

    .. math::
        \Omega(a) = \frac{\omega_m}{\omega_m + a\,(1 - \omega_m - \omega_l) + \omega_l\,a^3}.

    :math:`F_\omega` approximates the logarithmic derivative of the linear
    growth factor :math:`D_1` with respect to the scale factor :math:`a`,
    i.e.

    .. math::
        f \equiv \frac{d\ln(D_1)}{d\ln(a)}.


    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The linear growth rate :math:`F_\omega(a)`.
    '''
    omega_a = omega_m / (omega_m + a * (1 - omega_m - omega_l) + a**3 * omega_l)
    return np.power(omega_a, 5.0/9.0)  # Bernardeau et al. 2001, eq. 101a

In [ ]:
z = np.linspace(63, 0, 1000)
a = 1 / (1 + z)
F = F_omega(a, params['OMEGA_M'], params['OMEGA_L'])

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

ax = axes[0]
ax.plot(z, F, color='tab:red', lw=3)

ax = axes[1]
ax.plot(z, F, color='tab:red', lw=3)
ax.scatter(z[-1], F[-1], color='black', marker='x', s=12**2, lw=2, zorder=2)
ax.axvline(0, color='0.7', lw=1, ls='--')
ax.axhline(F[-1], color='green', lw=1, ls='--')
ax.text(0.05, 0.27, rf'Final $F = {F[-1]:.4f}$', color='green',
        fontsize=10, ha='left', va='center', transform=ax.transAxes)
ax.set_xlim(-0.2, 0.4)
ax.set_ylim(0.4, 0.7)

for ax in axes:
    # ax.set_xscale('log')
    ax.invert_xaxis()
    ax.set_xlabel(r'Redshift $z$', fontsize=12)
    ax.set_title('Linear growth rate factor', loc='left', fontsize=10)

plt.show()

In [ ]:
def F2_omega(a, omega_m, omega_l):
    r'''
    Computes the second-order growth rate factor for second-order
    Lagrangian perturbation theory corrections.

    This function returns the factor :math:`F2_\omega(a)`, defined by

    .. math::
        F2_\omega(a) = 2\,\left[\Omega(a)\right]^{\frac{4}{7}},

    where the effective matter density parameter :math:`\Omega(a)` is
    computed as

    .. math::
        \Omega(a) = \frac{\omega_m}{\omega_m + a\,(1 - \omega_m - \omega_l) + \omega_l\,a^3}.

    :math:`F2_\omega` is used in second-order Lagrangian perturbation theory
    to scale the second-order displacement field and its time derivative,
    thereby accounting for non-linear corrections to the growth of structure.

    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The second-order growth rate :math:`F2_\omega` evaluated at scale
        factor :math:`a`.
    '''
    omega_a = omega_m / (omega_m + a * (1 - omega_m - omega_l) + a**3 * omega_l)
    return 2 * np.power(omega_a, 6.0/11.0)  # Bernardeau et al. 2001, eq. 101b

In [ ]:
z = np.linspace(63, 0, 1000)
a = 1 / (1 + z)
F2 = F2_omega(a, params['OMEGA_M'], params['OMEGA_L'])

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

ax = axes[0]
ax.plot(z, F2, color='tab:red', lw=3)

ax = axes[1]
ax.plot(z, F2, color='tab:red', lw=3)
ax.scatter(z[-1], F2[-1], color='black', marker='x', s=12**2, lw=2, zorder=2)
ax.axvline(0, color='0.7', lw=1, ls='--')
ax.axhline(F2[-1], color='green', lw=1, ls='--')
ax.text(0.05, 0.22, rf'Final $F_2= {F2[-1]:.4f}$', color='green',
        fontsize=10, ha='left', va='center', transform=ax.transAxes)
ax.set_xlim(-0.2, 0.4)
ax.set_ylim(0.9, 1.5)

for ax in axes:
    # ax.set_xscale('log')
    ax.invert_xaxis()
    ax.set_xlabel(r'Redshift $z$', fontsize=12)
    ax.set_title('Second-order growth rate factor', loc='left', fontsize=10)
plt.show()

## Colossus and CAMB

In [ ]:
class ColossusCosmology:
    def __init__(self, *,
            H0=67.742, Om0=0.3099, Ob0=0.048891, Ol0=0.6901, sigma8=0.8105,
            ns=0.96822, Neff=3.046, w0=-1.0, wa=0.0, **kwargs):
        '''
        Wrapper to initialize a Colossus cosmology object.

        Parameters
        ----------
        H0 : float
            Hubble constant in km/s/Mpc.
        Om0 : float
            Total matter density parameter today divided by the critical density.
        Ob0 : float
            Baryon density parameter today divided by the critical density.
        Ol0 : float
            Dark energy density parameter today divided by the critical density.
        sigma8 : float
            RMS matter fluctuation amplitude at redshift 0.
        ns : float
            Scalar spectrum power-law index for k_pivot = 0.05 Mpc^-1.
        Neff : float
            Total effective number of massive and massless neutrinos.
        w0 : float
            Dark energy equation of state parameter at redshift 0.
        wa : float
            Dark energy equation of state parameter evolution.
        '''
        flat = np.isclose(1 - Ol0 - Om0, 0.0, rtol=1e-5, atol=1e-8)
        de_model = 'w0wa' if wa != 0.0 else 'w0' if w0 != -1.0 else 'lambda'
        self.cosmo = cosmology.setCosmology(
            'steps', H0=H0, Om0=Om0, Ob0=Ob0, sigma8=sigma8, ns=ns, Neff=Neff,
            w0=w0, wa=wa, flat=flat, de_model=de_model, **kwargs)
    
    def Dzplus0(self, z : float) -> float:
        '''
        Calculate the linear growth factor :math:`D_+(z)` normalized to
        1 at present.

        For :math:`w_0` and CPL dark energy, we use the Colossus
        implementation of Eq. (11) from Linder & Jenkins (2003).
        See paper at https://arxiv.org/pdf/astro-ph/0305286.pdf.
        
        Parameters
        ----------
        z : float
            Redshift(s) at which to compute the growth factor.
        
        Returns
        -------
        float or np.ndarray
            The linear growth factor D_+(z).
        '''
        Dzplus = self.cosmo.growthFactorUnnormalized(z)
        D0plus = self.cosmo.growthFactorUnnormalized(0.0)

        log.info(f'D_+({z=:.2f})/D_+(z=0) = {Dzplus / D0plus:.2e}')
        return Dzplus / D0plus

In [ ]:
z = 63
a = 1.0 / (1.0 + z)

cosmo = ColossusCosmology(Tcmb0=1e-6)
log.info(cosmo.cosmo.de_model)
_ = cosmo.Dzplus0(z)

cosmo = ColossusCosmology(Tcmb0=1e-6, w0=-0.94)
log.info(cosmo.cosmo.de_model)
_ = cosmo.Dzplus0(z)

cosmo = ColossusCosmology(Tcmb0=1e-6, w0=-0.94, wa=0.06)
log.info(cosmo.cosmo.de_model)
_ = cosmo.Dzplus0(z)

cosmo = ColossusCosmology(Tcmb0=1e-6, wa=0.06)
log.info(cosmo.cosmo.de_model)
_ = cosmo.Dzplus0(z)


## Power spectrum

In [ ]:
class CAMBCosmology:
    def __init__(self, *,
            H0=67.742, ombh2=0.022436, omch2=0.11914, omk=0.0, mnu=0.06,
            nnu=3.046, YHe=0.245421, zrei=7.89, TCMB=2.775, w0=-1.0, wa=0.0,
            nonlinear=False, halofit_version='mead2020', **kwargs):
        '''
        Wrapper to initialize a CAMB cosmology object.
        
        Parameters
        ----------
        H0 : float
            Hubble constant in km/s/Mpc.
        ombh2 : float
            Baryon density parameter today.
        omch2 : float
            Cold dark matter density parameter today.
        omk : float
            Curvature density today divided by the critical density.
        mnu : float
            Sum of active neutrino masses in eV.
        nnu : float
            Total effective number of massive and massless neutrinos.
        YHe : float
            Fraction of baryonic mass in helium. Set to `None` to be
            calculated internally for BBN consistency.
        TCMB : float
            CMB temperature in Kelvin.
        zrei : float
            Redshift at which the Universe is half reionized.
        w0 : float
            Dark energy equation of state parameter at redshift 0.
        wa : float
            Dark energy equation of state parameter evolution.
        nonlinear : bool
            If True, include non-linear corrections using Halofit.
        halofit_version : str
            Version of the Halofit model to use for non-linear corrections.
            Check ``camb.nonlinear.Halofit`` for available models.
        '''
        self.params = camb.CAMBparams()
        self.params.set_cosmology(
            H0=H0, ombh2=ombh2, omch2=omch2, omk=omk, mnu=mnu, nnu=nnu,
            YHe=YHe, TCMB=TCMB, zrei=zrei, **kwargs)
        if w0 != -1.0 or wa != 0.0:
            log.info(f'Using single fluid dark energy model with w0={w0} and wa={wa}')
            self.params.DarkEnergy = camb.dark_energy.DarkEnergyFluid()
            self.params.DarkEnergy.set_params(w=w0, wa=wa)

        if nonlinear:
            log.info(f'Using non-linear corrections with Halofit model `{halofit_version}`')
            self.params.NonLinear = camb.model.NonLinear_both
            self.params.NonLinearModel = camb.nonlinear.Halofit()
            self.params.NonLinearModel.set_params(halofit_version=halofit_version)
        else:
            log.info('Using linear theory only.')
            self.params.NonLinear = camb.model.NonLinear_none

    def get_sigma8(self, z=0, *, As=2.1064e-09, ns=0.96822, kmax=1.0):
        r'''
        Calculate the RMS matter fluctuation amplitude :math:`\sigma_8`
        at given redshift ``z`` using CAMB.
        
        Parameters
        ----------
        z : float
            Target redshift.
        As : float
            Comoving curvature power at :math:`k = 0.05\,\mathrm{Mpc}^{-1}`.
            This is the amplitude of the primordial power spectrum at large
            scales, typically set to match the observed :math:`\sigma_8`.
        ns : float
            Scalar spectral index.
        kmax : float
            Maximum wavenumber in :math:`h^{-1}\,\mathrm{Mpc}`.

        Returns
        -------
        float
            The RMS matter fluctuation amplitude :math:`\sigma_8` at
            redshift :math:`z`.
        '''
        self.params.InitPower.set_params(As=As, ns=ns)
        self.params.set_matter_power(redshifts=[z], kmax=kmax)
        results = camb.get_results(self.params)
        sigma8 = results.get_sigma8()[0]
        return sigma8
    
    def _rescale_As(self, target_sigma8, As=2.1064e-09, ns=0.96822, kmax=1.0):
        '''
        Rescale the matter fluctuation amplitude :math:`A_s` such that
        :math:`\sigma_8(z=0, A_s=\mathrm{new\_As}) = \mathrm{target\_sigma8}`.
        '''
        sigma8_now = self.get_sigma8(z=0, As=As, ns=ns, kmax=kmax)
        if target_sigma8 is None or np.isclose(sigma8_now, target_sigma8, rtol=1e-4):
            return sigma8_now, As
        scale = (target_sigma8 / sigma8_now)**2
        log.info(f'Rescaling matter-fluctuation amplitude by {scale:.3g}')
        As_new = As * scale
        sigma8_new = self.get_sigma8(z=0, As=As_new, ns=ns, kmax=kmax)
        return sigma8_new, As_new

    def get_spectrum(self, *,
            z=127, As=2.1064e-09, ns=0.96822, sigma8_init=None,
            kmin=0.01, kmax=1.0, npoints=512, component='delta_cdm'):
        r'''
        Calculate the matter power spectrum using CAMB.

        Parameters
        ----------
        z : float or list of float
            Redshifts at which the linear power spectrum is calculated.
        As : float
            Comoving curvature power at :math:`k = 0.05\,\mathrm{Mpc}^{-1}`.
            This is the amplitude of the primordial power spectrum at large
            scales, typically set to match the observed :math:`\sigma_8`.
        ns : float
            Scalar spectral index.
        sigma8_init : float
            Rescale the matter power spectrum to the given :math:`\sigma_8`
            value.
        kmin : float
            Minimum wavenumber in :math:`h^{-1}\,\mathrm{Mpc}`.
        kmax : float
            Maximum wavenumber in :math:`h^{-1}\,\mathrm{Mpc}`.
        npoints : int
            Number of wavenumber points.
        component : str
            The component of the power spectrum to return. Options are:
            'delta_tot' for total matter, 'delta_cdm' for cold dark matter,
            'delta_baryon' for baryonic matter, etc. See CAMB documentation
            for more details.
        '''
        # Optional rescaling of the `As` amplitude to match a desired sigma8
        sigma8, As = self._rescale_As(sigma8_init, As=As, ns=ns, kmax=kmax)
        log.info(f'Value for matter fluctuation amplitude used: {sigma8 = :.4f}')

        # Calculating P(k) at redshift `z`
        self.params.set_matter_power(redshifts=np.atleast_1d(z).tolist(), kmax=kmax)
        results = camb.get_results(self.params)
        kh, _, pk = results.get_matter_power_spectrum(
            minkh=kmin, maxkh=kmax, npoints=npoints, var1=component, var2=component)
        pk3 = pk * kh**3/(2*np.pi**2)  # Save (log(kh), log(pk3)).T for StePS/Gadget
        return kh, pk, pk3

In [ ]:
kmin = 1/np.min(params['LBOX'])
kmax = 1.0
npoints = 2048

In [ ]:
cosmo_camb = CAMBCosmology(
    H0=params['H0'], ombh2=params['OMBH2'], omch2=params['OMCH2'],
    omk=params.get('OMK', 0.0), mnu=params['MNU'], nnu=params['NNU'],
    YHe=params['YHE'], TCMB=params['TCMB'], zrei=params['ZREI'],
    w0=params['W0'], wa=params['WA'], nonlinear=False)
kh, pk_l, pk3_l = cosmo_camb.get_spectrum(
    z=0, As=params['AS'], ns=params['NS'], sigma8_init=None,
    kmin=kmin, kmax=kmax, npoints=npoints)

cosmo_camb = CAMBCosmology(
    H0=params['H0'], ombh2=params['OMBH2'], omch2=params['OMCH2'],
    omk=params.get('OMK', 0.0), mnu=params['MNU'], nnu=params['NNU'],
    YHe=params['YHE'], TCMB=params['TCMB'], zrei=params['ZREI'],
    w0=params['W0'], wa=params['WA'], nonlinear=True)
kh, pk_nl, pk3_nl = cosmo_camb.get_spectrum(
    z=0, As=params['AS'], ns=params['NS'], sigma8_init=None,
    kmin=kmin, kmax=kmax, npoints=npoints)

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*4.5, nr*4), dpi=120)

cosmo_colossus = ColossusCosmology(
    H0=params['H0'], Om0=params['OMEGA_M'], Ob0=params['OMEGA_B'],
    Ol0=params['OMEGA_L'], sigma8=params['SIGMA8'], ns=params['NS'],
    Neff=params['NNU'], w0=params['W0'], wa=params['WA'], Tcmb0=1e-6)

zs = [127, 63, 31]
ls = ['-', '--', '-.']
for z_i, ls_i in zip(zs, ls):
    Dzplus0 = cosmo_colossus.Dzplus0(z_i)
    ax = axes[0]
    y_l, y3_l = pk_l*Dzplus0**2, pk3_l*Dzplus0**2
    y_nl, y3_nl = pk_nl*Dzplus0**2, pk3_nl*Dzplus0**2
    ax.loglog(kh, y_l[0], color='0.3', lw=2, ls=ls_i)
    ax.loglog(kh, y_nl[0], color='tab:red', lw=2, ls=ls_i)
    ax.set_title(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', loc='left', fontsize=10)

    ax = axes[1]
    ax.loglog(kh, y3_l[0], color='0.3', lw=2, ls=ls_i)
    ax.loglog(kh, y3_nl[0], color='tab:red', lw=2, ls=ls_i)
    ax.set_title(r'$k^3 P(k) / \left( 2\pi^2 \right)$', loc='left', fontsize=10)

handles = [Line2D([0], [0], label=f'linear', color='0.3', lw=2),
           Line2D([0], [0], label=f'non-linear', color='tab:red', lw=2)]
for ax in axes:
    ax.set_box_aspect(1)
    ax.set_xlabel(r'$k\,[h/\mathrm{Mpc}]$', fontsize=10)
    ax.legend(handles=handles, fontsize=10, frameon=False, facecolor='none')

plt.show()